<a href="https://colab.research.google.com/github/krimits/hotel-review-nlp/blob/experiment/bilstm-threshold-tuning/hotel-review-nlp/notebooks/04_bilstm_threshold_tuning_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Seed-100 BiLSTM — dev-only threshold tuning

## Goal

Run one controlled follow-up to the final **unweighted** seed-100 BiLSTM. Training remains unchanged. The only experimental change is that the positive-class decision threshold is selected on the development set using macro-F1, and the frozen test set is evaluated once afterward.

This notebook imports the implementation from the repository; it does not maintain a second copied training loop.

## Setup

Use a GPU runtime (`Runtime → Change runtime type → T4 GPU`). The cell below clones the experiment branch into a separate directory and installs the package plus W&B tracking.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPOSITORY_URL = "https://github.com/krimits/hotel-review-nlp.git"
BRANCH = "experiment/bilstm-threshold-tuning"
CHECKOUT_DIR = Path("/content/hotel-review-threshold")

if not CHECKOUT_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPOSITORY_URL, str(CHECKOUT_DIR)],
        check=True,
    )

PROJECT_DIR = CHECKOUT_DIR / "hotel-review-nlp"
os.chdir(PROJECT_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[tracking]"], check=True)

print("Project:", PROJECT_DIR)
print("Branch:", BRANCH)
subprocess.run(["git", "rev-parse", "--short", "HEAD"], check=True)

### Connect W&B

Authenticate with your own W&B account. Do not paste an API key into a notebook cell that will be saved to GitHub.

In [ ]:
import wandb

wandb.login()

## Data

### Key assumptions

- `train.parquet`, `dev.parquet`, and `test.parquet` are the exact frozen splits used by the earlier seed-100 run.
- No split is regenerated with seed 100; changing the split would invalidate the controlled comparison.
- If the previous Colab checkout still exists, the next cell copies the splits from it. Otherwise it opens an upload dialog for the three parquet files.

In [ ]:
from pathlib import Path
import shutil

required_files = ("train.parquet", "dev.parquet", "test.parquet")
destination = PROJECT_DIR / "data" / "processed"
destination.mkdir(parents=True, exist_ok=True)

candidate_directories = [
    Path("/content/hotel-review-nlp/hotel-review-nlp/data/processed"),
    Path("/content/data/processed"),
]

if not all((destination / name).exists() for name in required_files):
    for candidate in candidate_directories:
        if all((candidate / name).exists() for name in required_files):
            for name in required_files:
                shutil.copy2(candidate / name, destination / name)
            print("Copied frozen splits from:", candidate)
            break

missing = [name for name in required_files if not (destination / name).exists()]
if missing:
    from google.colab import files

    print("Upload the exact frozen split files:", missing)
    uploaded = files.upload()
    for uploaded_name, content in uploaded.items():
        basename = Path(uploaded_name).name
        if basename in required_files:
            (destination / basename).write_bytes(content)

missing = [name for name in required_files if not (destination / name).exists()]
if missing:
    raise FileNotFoundError(f"Still missing frozen split files: {missing}")

print("All frozen splits are available.")

### Validate the frozen splits

In [ ]:
import pandas as pd

split_summary = []
for split_name in ("train", "dev", "test"):
    frame = pd.read_parquet(destination / f"{split_name}.parquet")
    counts = frame["label"].value_counts().to_dict()
    split_summary.append(
        {
            "split": split_name,
            "rows": len(frame),
            "negative": int(counts.get("negative", 0)),
            "positive": int(counts.get("positive", 0)),
        }
    )

split_summary = pd.DataFrame(split_summary)
display(split_summary)

expected_rows = {"train": 118_990, "dev": 14_872, "test": 13_278}
observed_rows = dict(zip(split_summary["split"], split_summary["rows"], strict=True))
assert observed_rows == expected_rows, (observed_rows, expected_rows)

## Run the controlled experiment

The training objective remains ordinary, unweighted cross-entropy. At each epoch, threshold selection uses only the dev set. The checkpoint and threshold with the best dev macro-F1 are then applied once to the test set.

In [ ]:
from reviewnlp.baselines.bilstm import train_bilstm

final_test_metrics = train_bilstm("configs/bilstm_seed100_threshold.yaml")
final_test_metrics

## Results

In [ ]:
import json

metrics_path = PROJECT_DIR / "runs" / "bilstm_seed100_threshold" / "metrics.json"
with metrics_path.open(encoding="utf-8") as file:
    results = json.load(file)

comparison = pd.DataFrame(
    [
        {"decision_rule": "threshold = 0.5", **results["test_threshold_0_5"]},
        {
            "decision_rule": f"dev-selected threshold = {results['selected_threshold']:.3f}",
            **results["test"],
        },
    ]
)[["decision_rule", "accuracy", "macro_f1", "macro_precision", "macro_recall", "weighted_f1"]]

display(comparison)
print("Selected only on dev:", results["selected_threshold"])
print("Default confusion matrix:", results["test_threshold_0_5"]["confusion_matrix"])
print("Selected-threshold confusion matrix:", results["test"]["confusion_matrix"])

## Checks

In [ ]:
import numpy as np

run_directory = metrics_path.parent
for artifact_name in (
    "dev_logits.npy",
    "dev_labels.npy",
    "test_logits.npy",
    "test_labels.npy",
    "metrics.json",
):
    assert (run_directory / artifact_name).exists(), artifact_name

dev_labels = np.load(run_directory / "dev_labels.npy")
test_labels = np.load(run_directory / "test_labels.npy")
assert len(dev_labels) == 14_872
assert len(test_labels) == 13_278
assert results["selection_metric"] == "dev_macro_f1"

print("All result artifacts and split sizes are valid.")

## Next Steps

In [ ]:
default_macro_f1 = results["test_threshold_0_5"]["macro_f1"]
tuned_macro_f1 = results["test"]["macro_f1"]
delta_pp = 100 * (tuned_macro_f1 - default_macro_f1)

if delta_pp > 0:
    decision = "Retain the dev-selected threshold."
elif delta_pp < 0:
    decision = "Retain the default threshold of 0.5."
else:
    decision = "No measurable difference; retain 0.5 for simplicity."

print(f"Test macro-F1 change: {delta_pp:+.2f} percentage points")
print(decision)
print("Use these executed values to update the PR and stakeholder summary.")